## Real vs Fake Faces

We are going to `Fake-Vs-Real-Faces`. The Dataset contains two types of image which are: `Fake faces` image collected from the website [thispersondoesnotexist.com](thispersondoesnotexist.com). , `Real faces` images are re collected through the API of the website [Unsplash](https://unsplash.com/) and then the faces are cropped out of using OpenCV library.

Here some additional information about the dataset:
- Total number of images: 1288
- Number of "Fake" faces: 700
- Number of "Real" faces: 589

### 1. Import Libraries 

In [1]:
import torch 
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder # Stream data from images stored in folders
import torch.optim as optim
import torch.nn.functional as F

import os # allow s to access files
import numpy as np
from PIL import Image # Allows us to Load Images
from collections import Counter # Utility function to give us the counts of unique items in an iterable

### 2. Load and Process the data

The first `dataloader` we will build will be the `Real vs Fake Faces` dataset. which is  An **abstract class** that represents our data (images, text, tabular, etc.)

In [2]:
torch.manual_seed(42)


class RealVsFake(Dataset):

    # we overriding these method in Dataset class after inhertianing from it

    def __init__(self, path_to_folder):

        path_to_real = os.path.join(path_to_folder, 'real')
        path_to_fake = os.path.join(path_to_folder, 'fake')

        # list of files inside the current folder
        real_files = os.listdir(path_to_real)
        fake_files = os.listdir(path_to_fake)

        # the absolute path for each image or file
        path_to_real_files = [os.path.join(path_to_real, file) for file in real_files]
        path_to_fake_files = [os.path.join(path_to_fake, file) for file in fake_files]

        self.training_files = path_to_real_files + path_to_fake_files

        self.real_label = 0
        self.fake_label = 1

        # transform convert an image to tensor and scale the pixel value [0, 255] to be between [0,1] to be between [0, 1]
        self.transform = transforms.Compose([
                        transforms.Resize((224, 224)),
                        transforms.RandomHorizontalFlip(),
                        transforms.RandomRotation(10),
                        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
                        ])

    def __len__(self):
        return len(self.training_files)

    def __getitem__(self, idx):

        # access the image path from paths list
        path_to_image = self.training_files[idx]

        # if real word in the path set label to real label
        if "real" in path_to_image:
            label = self.real_label
        else:
            label = self.fake_label

                
        # load the image using PIL module and convert it to RGB image
        image = Image.open(path_to_image).convert("RGB")
    
            
        image = self.transform(image)
        return image, label

In [3]:
path_to_folder = "./data/"
dataset = RealVsFake(path_to_folder)
BATCH_SIZE = 128

realvsfakeloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

In [4]:
for images, labels in realvsfakeloader:
    print(images.shape)
    print(labels)
    break

torch.Size([128, 3, 224, 224])
tensor([1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
        1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0,
        0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1,
        1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
        1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1,
        1, 0, 0, 0, 0, 1, 1, 1])


### 3. Split the data

Split the data into trianing and validation and testing

In [5]:
num_train_samples = int(0.9*len(dataset))
num_val_samples = int(0.1*len(dataset))
num_test_samples = len(dataset) - num_train_samples - num_val_samples



train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, lengths=[num_train_samples, num_val_samples, num_test_samples])

realvsfakeloader_train = DataLoader(train_dataset, batch_size=16, shuffle=True)
realvsfakeloader_val = DataLoader(val_dataset, batch_size=16, shuffle=True)
realvsfakeloader_test = DataLoader(test_dataset, batch_size=16, shuffle=True)

### 4. Build the Model

After processing the let's build the model

In [58]:
class RealVsFakeModel(nn.Module):

    def __init__(self, in_channels, num_classes):
        super(RealVsFakeModel, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, 20, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.p1  = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(20, 40, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.p2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(40, 60, 3, padding=1)
        self.relu3 = nn.ReLU()
        self.p3 = nn.MaxPool2d(2, 2)
        
        self.conv4 = nn.Conv2d(60, 40, 3, padding=1)
        self.relu4 = nn.ReLU()
        self.p4  = nn.MaxPool2d(2, 2)
        
        self.conv5 = nn.Conv2d(40, 20, 3, padding=1)
        self.relu5 = nn.ReLU()
        self.p5 = nn.MaxPool2d(2, 2)

       
        self.fc1 = nn.Linear(20 * 7 * 7, out_features=128)
        self.relu6 = nn.ReLU()
        
        self.fc2 = nn.Linear(128, 256)
        self.relu7 = nn.ReLU()

        self.fc3 = nn.Linear(256, 1) 
        self.output = nn.Sigmoid() 

    def forward(self, x):
        x = self.p1(self.relu1(self.conv1(x)))
        x = self.p2(self.relu2(self.conv2(x)))
        x = self.p3(self.relu3(self.conv3(x)))
        x = self.p4(self.relu4(self.conv4(x)))
        x = self.p5(self.relu5(self.conv5(x)))

        x = x.view(x.size(0), -1)
        
        x = self.relu6(self.fc1(x))
        x = self.relu7(self.fc2(x))
        x = self.fc3(x)
        x = self.output(x)

        return x

        

In [54]:
class RealVsFakeModel(nn.Module):

    def __init__(self, in_channels, num_classes):
        super(RealVsFakeModel, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, 16, 5, padding=2)
        self.relu1 = nn.ReLU()
        self.p1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(16, 32, 5, padding=2)
        self.relu2 = nn.ReLU()
        self.p2 = nn.MaxPool2d(2, 2)
        
  

        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.relu3 = nn.ReLU()
        
        self.fc2 = nn.Linear(128, 256)
        self.relu4 = nn.ReLU()

        self.fc3 = nn.Linear(256, 1) 
        self.output = nn.Sigmoid() 
         

    def forward(self, x):
        # Block 1
        x = self.p1(self.relu1(self.conv1(x)))
        x = self.p2(self.relu2(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        x = self.relu4(x)
        x = self.fc3(x)
        x = self.output(x)
        return x

        

In [59]:
LOSS = nn.BCEWithLogitsLoss()
LR = 0.05
EPOCH = 5

In [60]:
model = RealVsFakeModel(3, 1)
optimizer = optim.Adam(model.parameters(), lr=LR)

In [61]:
for epoch in range(EPOCH):
    print(f"Training Epoch {epoch+1}")

    losses = []
    for images, labels in realvsfakeloader_train:
        forward_out = model(images)
        labels = labels.unsqueeze(1).float()
        loss = LOSS(forward_out, labels)
        loss.backward() # backpropagation
        optimizer.step() # update the weight
        optimizer.zero_grad() # set the gradient to zero
        losses.append(loss.item()) # mean of loss over each batch
        
    print(f"Loss: {np.mean(losses)}") 

    
    model.eval()
    val_losses = []
    
    with torch.no_grad():  # Disable gradient calculation during validation
        for images, labels in realvsfakeloader_val:
            forward_out = model(images)
            labels = labels.unsqueeze(1).float()
    
            loss = LOSS(forward_out, labels)
            val_losses.append(loss.item())
    
    avg_val_loss = np.mean(val_losses)
    print(f"Validation Loss: {avg_val_loss:.4f}\n")




Training Epoch 1
Loss: 0.7636591587981133
Validation Loss: 0.8367

Training Epoch 2
Loss: 0.7618918892455427
Validation Loss: 0.8367

Training Epoch 3
Loss: 0.763604220053921
Validation Loss: 0.8367

Training Epoch 4
Loss: 0.7653165459632874
Validation Loss: 0.8367

Training Epoch 5
Loss: 0.7618918847547819
Validation Loss: 0.8367



In [76]:
model = torch.load(PATH, weights_only=False)

In [77]:
print(f"Loss: {np.mean(losses)}") 

    
model.eval()
val_losses = []
    
with torch.no_grad():  # Disable gradient calculation during validation
    for images, labels in realvsfakeloader_test:
        forward_out = model(images)
        labels = labels.unsqueeze(1).float()
    
        loss = LOSS(forward_out, labels)
        val_losses.append(loss.item())
    
    avg_val_loss = np.mean(val_losses)
print(f"Validation Loss: {avg_val_loss:.4f}\n")


Loss: 0.7618918847547819
Validation Loss: 0.3133

